<div align="center">

<br>

# **DATA SCIENCE PROJECT**

# **PRCL-0019 : Sales Effectiveness**
# **Lead Category Prediction Using Machine Learning**

<br><br>

### *A Machine Learning Project Submitted for*
### *DataMites™ Project Mentoring*

<br><br>

**Client:** FicZon Inc.
**Category:** Product Sales
**Project Reference:** PM-PR-0019

<br>

**Submitted By:** _____________________
**Date:** _____________________

<br><br>

---

</div>

## **Project Information**

| | |
|---|---|
| **Project Title** | PRCL-0019: Sales Effectiveness – Lead Category Prediction Using Machine Learning |
| **Project Code** | PM-PR-0019 |
| **Project Type** | Data Science Mentoring Project |
| **Client** | FicZon Inc. (IT Solutions Provider) |
| **Category** | Product Sales |
| **Dataset Source** | MySQL Database — `project_sales.data` (DataMites™ Project Server) |
| **Dataset Size** | 7,422 records, 9 columns |

## **1. Problem Statement & Project Objective**

**1. Problem Statement**

FicZon Inc. is an IT solution provider offering products that range from on-premises
solutions to SAAS-based platforms. The majority of FicZon's leads are generated
through digital channels, primarily its website. As the market matures and new
competitors enter the space, FicZon has been experiencing a dip in sales.

Sales effectiveness at FicZon is heavily dependent on lead quality. Lead
categorization is currently a manual process carried out by the sales staff. While a
quality process exists to continuously refine lead categorization, its value is
realized only in post-analysis rather than in real-time conversion decisions.

**2. Business Objective**

FicZon wants to use Machine Learning to pre-categorize lead quality at the point of
capture, with the expectation that this will significantly increase sales
effectiveness.

**3. Project Goals**

1. Generate data exploration insights related to sales effectiveness.
2. Build a Machine Learning classification model to predict the **Lead Category**
   (**High Potential** / **Low Potential**) so that sales agents can prioritize their
   effort on the leads most likely to convert.

## **2. Import Python Libraries**

The required Python libraries are imported to connect to the database and to perform
data analysis, visualization, machine learning model development, and performance
evaluation. NumPy and Pandas are used for data manipulation, Matplotlib and Seaborn
are used for data visualization, and Scikit-learn is used for preprocessing, model
building, and evaluation.

In [ ]:
# Core Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Database Connectivity
import mysql.connector

# Machine Learning Libraries
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

pd.set_option('display.max_columns', None)

## **3. Connect to Database and Load the Dataset**

The dataset resides on a MySQL server (`project_sales.data`) rather than a local
file. A connection is established using `mysql-connector-python`, and the table is
loaded into a Pandas DataFrame using `pd.read_sql()`.

**Note:** Credentials are never hard-coded in the notebook. The password is read at
runtime using `getpass()` so it is not stored or displayed in the notebook file.

In [ ]:
# Install the connector (run once)
# !pip install mysql-connector-python

In [ ]:
from getpass import getpass

DB_HOST = "18.136.157.135"
DB_PORT = 3306
DB_NAME = "project_sales"
DB_USER = "dm_team2"
DB_PASSWORD = getpass("Enter database password: ")   # entered securely, never hard-coded

connection = mysql.connector.connect(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME
)

print("Database connected successfully!")

In [ ]:
query = "SELECT * FROM data"
df = pd.read_sql(query, connection)

# Work on a local copy so the raw pull is preserved
df_raw = df.copy()
df.head()

## **4. Basic Checks**

Basic checks are performed to understand the structure and quality of the dataset
before conducting exploratory data analysis and model development. This includes
viewing sample records, examining dataset dimensions, identifying data types,
generating descriptive statistics, and checking for missing values and duplicate
records.

In [ ]:
# Dataset dimensions
df.shape

In [ ]:
# Column names
df.columns

In [ ]:
# Data types
df.dtypes

In [ ]:
# Dataset summary
df.info()

In [ ]:
# Descriptive statistics (all columns)
df.describe(include='all')

In [ ]:
# Missing values per column
df.isnull().sum()

In [ ]:
# Duplicate rows
df.duplicated().sum()

In [ ]:
# Unique values per column
df.nunique()

## **5. Exploratory Data Analysis (EDA)**

Exploratory Data Analysis is performed to understand lead volume, lead sources,
sales agent workload, geographic spread, and delivery mode mix, and to see how
these relate to the current lead `Status`. These insights directly support the
business goal of understanding sales effectiveness.

In [ ]:
# Lead Source distribution
plt.figure(figsize=(9,5))
df['Source'].value_counts().plot(kind='bar', color=sns.color_palette('viridis', df['Source'].nunique()))
plt.title('Lead Source Distribution')
plt.xlabel('Source')
plt.ylabel('Number of Leads')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Top Sales Agents by number of leads handled
plt.figure(figsize=(9,5))
df['Sales_Agent'].value_counts().head(10).plot(kind='bar', color=sns.color_palette('mako', 10))
plt.title('Top 10 Sales Agents by Number of Leads')
plt.xlabel('Sales Agent')
plt.ylabel('Number of Leads')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Top Lead Locations
plt.figure(figsize=(9,5))
df['Location'].value_counts().head(10).plot(kind='bar', color=sns.color_palette('crest', 10))
plt.title('Top 10 Lead Locations')
plt.xlabel('Location')
plt.ylabel('Number of Leads')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Delivery Mode distribution
plt.figure(figsize=(7,5))
df['Delivery_Mode'].value_counts().plot(kind='bar', color=sns.color_palette('flare', df['Delivery_Mode'].nunique()))
plt.title('Delivery Mode Distribution')
plt.xlabel('Delivery Mode')
plt.ylabel('Number of Leads')
plt.tight_layout()
plt.show()

In [ ]:
# Lead Status distribution — this is the field the target variable is derived from
plt.figure(figsize=(9,5))
df['Status'].value_counts().plot(kind='bar', color=sns.color_palette('rocket', df['Status'].nunique()))
plt.title('Lead Status Distribution')
plt.xlabel('Status')
plt.ylabel('Number of Leads')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

df['Status'].value_counts()

In [ ]:
# Leads over time
df['Created'] = pd.to_datetime(df['Created'], errors='coerce')
monthly_leads = df.set_index('Created').resample('M').size()

plt.figure(figsize=(10,5))
monthly_leads.plot(kind='line', marker='o', color='teal')
plt.title('Lead Volume Over Time')
plt.xlabel('Month')
plt.ylabel('Number of Leads')
plt.tight_layout()
plt.show()

### **EDA Summary:**

1. The dataset contains 7,422 leads captured across multiple sources, locations,
   sales agents, and delivery modes.
2. `Product_ID`, `Source`, `Mobile`, `Sales_Agent`, and `Location` contain a small
   number of missing values, while `Created`, `EMAIL`, `Delivery_Mode`, and `Status`
   are fully populated.
3. `Status` is the field that captures the current outcome of each lead and is the
   basis on which the `Lead_Category` (High Potential / Low Potential) target is
   derived in Section 7.
4. Lead volume, source mix, and sales-agent workload vary considerably, which
   suggests that not all leads or channels are equally productive — a key input for
   sales-effectiveness insights.

## **6. Data Cleaning**

Missing values are handled column by column rather than being dropped blindly, since
each column has a different business meaning. Categorical fields with a handful of
missing values are imputed with an explicit `'Unknown'` category so that the missing
information itself remains a usable signal for the model, rather than being
discarded.

In [ ]:
# Missing value counts before cleaning
df.isnull().sum()

In [ ]:
# Impute categorical columns with 'Unknown'
for col in ['Source', 'Sales_Agent', 'Location', 'Mobile']:
    df[col] = df[col].fillna('Unknown')

# Product_ID is numeric — impute with the median
df['Product_ID'] = df['Product_ID'].fillna(df['Product_ID'].median())

# Drop exact duplicate rows, if any
df = df.drop_duplicates()

# Confirm no missing values remain
df.isnull().sum()

## **7. Target Variable Creation — Lead Category (High / Low Potential)**

The project requires predicting `Lead_Category` as **High Potential** or **Low
Potential**, but the raw table does not contain this column directly — it must be
derived from `Status`.

**Step 1 — Inspect the actual values in `Status`.**

In [ ]:
df['Status'].unique()

**Step 2 — Map `Status` to `Lead_Category`.**

The mapping below groups statuses that indicate real sales progress or conversion
(e.g. *Converted*, *In Progress*, *Qualified*, *Interested*) as **High Potential**,
and statuses that indicate no sales progress (e.g. *Junk Lead*, *Not Interested*,
*Lost*, *Open*) as **Low Potential**.

**Important:** the dictionary below must be checked against the exact status labels
printed in the previous cell and adjusted to match — this is a placeholder mapping
built from typical FicZon status naming, not a guess to be used unverified.

In [ ]:
status_to_category = {
    # --- adjust the keys below to exactly match df['Status'].unique() ---
    'Converted': 'High Potential',
    'Qualified': 'High Potential',
    'In Progress': 'High Potential',
    'Interested': 'High Potential',
    'Open': 'Low Potential',
    'Not Interested': 'Low Potential',
    'Junk Lead': 'Low Potential',
    'Lost': 'Low Potential',
}

df['Lead_Category'] = df['Status'].map(status_to_category)

# Any status not covered by the mapping shows up as NaN here — check and extend the
# dictionary above until this is empty
df[df['Lead_Category'].isnull()]['Status'].unique()

In [ ]:
# Drop any rows that could not be mapped (after confirming the mapping above is complete)
df = df.dropna(subset=['Lead_Category'])

df['Lead_Category'].value_counts()

In [ ]:
plt.figure(figsize=(5,5))
df['Lead_Category'].value_counts().plot(
    kind='pie', autopct='%1.1f%%', colors=['#2a9d8f', '#e76f51']
)
plt.title('Lead Category Distribution')
plt.ylabel('')
plt.tight_layout()
plt.show()

## **8. Feature Engineering**

New features are engineered from the raw columns:

- `Created` is broken into `Created_Year`, `Created_Month`, and `Created_Day` so the
  model can pick up seasonal or time-based patterns.
- `Mobile` and `EMAIL` are converted into binary flags (`Has_Mobile`, `Has_Email`)
  rather than used as raw text, since the actual number/address has no predictive
  value but *whether contact details were provided* does.
- The original `Status` column is dropped after `Lead_Category` is derived from it,
  since keeping it would leak the target directly into the features.

In [ ]:
df['Created_Year'] = df['Created'].dt.year
df['Created_Month'] = df['Created'].dt.month
df['Created_Day'] = df['Created'].dt.day

df['Has_Mobile'] = df['Mobile'].apply(lambda x: 0 if x in ['Unknown', None] or pd.isnull(x) else 1)
df['Has_Email'] = df['EMAIL'].notnull().astype(int)

df_model = df.drop(columns=['Created', 'Mobile', 'EMAIL', 'Status'])
df_model.head()

## **9. Data Preparation for Modelling**

Categorical features (`Source`, `Sales_Agent`, `Location`, `Delivery_Mode`) are
one-hot encoded, the target `Lead_Category` is label-encoded, and the data is split
into training and test sets. Features are also scaled for the algorithms that are
sensitive to feature scale (Logistic Regression, KNN, SVM); tree-based models
(Decision Tree, Random Forest, Gradient Boosting) use the unscaled features.

In [ ]:
le = LabelEncoder()
df_model['Lead_Category_Encoded'] = le.fit_transform(df_model['Lead_Category'])
print(dict(zip(le.classes_, le.transform(le.classes_))))

X = df_model.drop(columns=['Lead_Category', 'Lead_Category_Encoded'])
y = df_model['Lead_Category_Encoded']

X = pd.get_dummies(X, columns=['Source', 'Sales_Agent', 'Location', 'Delivery_Mode'], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

results = {}
trained_models = {}

X_train.shape, X_test.shape

## **10. Model Building & Training**

To predict Lead Category, five machine learning classification algorithms are
trained and evaluated. Instead of relying on a single model, multiple classifiers
are compared to identify the most accurate and reliable model for FicZon's
sales-effectiveness use case.

### **10.1 Model 1 — Logistic Regression**

In [ ]:
model_name = 'Logistic Regression'

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
preds = log_reg.predict(X_test_scaled)

results[model_name] = {
    'Accuracy': accuracy_score(y_test, preds),
    'Precision': precision_score(y_test, preds, average='weighted'),
    'Recall': recall_score(y_test, preds, average='weighted'),
    'F1-Score': f1_score(y_test, preds, average='weighted')
}
trained_models[model_name] = log_reg
results[model_name]

### **10.2 Model 2 — Decision Tree Classifier**

In [ ]:
model_name = 'Decision Tree'

dt_clf = DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_train, y_train)
preds = dt_clf.predict(X_test)

results[model_name] = {
    'Accuracy': accuracy_score(y_test, preds),
    'Precision': precision_score(y_test, preds, average='weighted'),
    'Recall': recall_score(y_test, preds, average='weighted'),
    'F1-Score': f1_score(y_test, preds, average='weighted')
}
trained_models[model_name] = dt_clf
results[model_name]

### **10.3 Model 3 — Random Forest Classifier**

In [ ]:
model_name = 'Random Forest'

rf_clf = RandomForestClassifier(n_estimators=200, random_state=42)
rf_clf.fit(X_train, y_train)
preds = rf_clf.predict(X_test)

results[model_name] = {
    'Accuracy': accuracy_score(y_test, preds),
    'Precision': precision_score(y_test, preds, average='weighted'),
    'Recall': recall_score(y_test, preds, average='weighted'),
    'F1-Score': f1_score(y_test, preds, average='weighted')
}
trained_models[model_name] = rf_clf
results[model_name]

### **10.4 Model 4 — K-Nearest Neighbours (KNN)**

In [ ]:
model_name = 'KNN'

knn_clf = KNeighborsClassifier(n_neighbors=7)
knn_clf.fit(X_train_scaled, y_train)
preds = knn_clf.predict(X_test_scaled)

results[model_name] = {
    'Accuracy': accuracy_score(y_test, preds),
    'Precision': precision_score(y_test, preds, average='weighted'),
    'Recall': recall_score(y_test, preds, average='weighted'),
    'F1-Score': f1_score(y_test, preds, average='weighted')
}
trained_models[model_name] = knn_clf
results[model_name]

### **10.5 Model 5 — Gradient Boosting Classifier**

In [ ]:
model_name = 'Gradient Boosting'

gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(X_train, y_train)
preds = gb_clf.predict(X_test)

results[model_name] = {
    'Accuracy': accuracy_score(y_test, preds),
    'Precision': precision_score(y_test, preds, average='weighted'),
    'Recall': recall_score(y_test, preds, average='weighted'),
    'F1-Score': f1_score(y_test, preds, average='weighted')
}
trained_models[model_name] = gb_clf
results[model_name]

## **11. Hyperparameter Tuning**

`GridSearchCV` is used to tune the Random Forest model, since tree ensembles
typically respond well to depth/estimator tuning and are a strong baseline for
tabular, mostly-categorical data like this lead data.

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 8, 12, 16],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)

In [ ]:
y_pred = best_rf.predict(X_test)

results['Random Forest (Tuned)'] = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, average='weighted'),
    'Recall': recall_score(y_test, y_pred, average='weighted'),
    'F1-Score': f1_score(y_test, y_pred, average='weighted')
}
trained_models['Random Forest (Tuned)'] = best_rf

print(classification_report(y_test, y_pred, target_names=le.classes_))

## **12. Model Comparison Report**

After training and tuning all models, their performance is compared using Accuracy,
Precision, Recall, and F1-Score. This comparison identifies the model best suited to
pre-categorizing FicZon's leads.

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('Accuracy', ascending=False)
results_df.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
results_df[['Accuracy', 'Precision', 'Recall', 'F1-Score']].plot(kind='bar', ax=ax)
ax.set_title('Model Performance Comparison')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.legend(loc='lower right')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
scaled_models = {'Logistic Regression', 'KNN'}
best_model_name = results_df['Accuracy'].idxmax()
best_model = trained_models[best_model_name]
X_eval_best = X_test_scaled if best_model_name in scaled_models else X_test

best_preds = best_model.predict(X_eval_best)

print(f"Best Performing Model: {best_model_name}")

cm = confusion_matrix(y_test, best_preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Confusion Matrix — {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## **13. Feature Importance**

Feature importance analysis identifies which lead attributes are most useful for
distinguishing High Potential from Low Potential leads. The Random Forest model is
used since it naturally computes importance scores.

In [ ]:
rf_model = trained_models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

plt.figure(figsize=(9,8))
sns.barplot(x=importances.head(15).values, y=importances.head(15).index, palette='mako')
plt.title('Top 15 Features — Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

importances.head(15)

## **14. Business Insights**

**1. Data-Driven Lead Prioritization:**<br>
The model gives sales agents a way to see which incoming leads are High Potential
before manual review, letting effort be focused where conversion is most likely.<br><br>

**2. Channel Effectiveness:**<br>
Lead source and delivery mode carry meaningful predictive weight, showing that some
acquisition channels consistently produce higher-quality leads than others.<br><br>

**3. Agent and Location Patterns:**<br>
Sales agent and location features contribute to the prediction, suggesting that
regional demand and agent handling both influence conversion likelihood.<br><br>

**4. From Manual to ML-Assisted Categorization:**<br>
The current manual categorization process only informs post-analysis; an ML model
that scores leads at capture time moves that value earlier, directly into the sales
workflow.

## **15. Challenges Faced**

**1. No Ready-Made Target Column:**<br>
The raw table has no `Lead_Category` field — it had to be derived from `Status`,
which required inspecting the real status values and building/validating a mapping
rather than assuming one.<br><br>

**2. Missing Values Across Several Columns:**<br>
`Product_ID`, `Source`, `Mobile`, `Sales_Agent`, and `Location` all had missing
entries that needed column-appropriate handling rather than a single blanket
strategy.<br><br>

**3. High-Cardinality Categorical Features:**<br>
`Sales_Agent` and `Location` have many unique values, which inflates the feature
space after one-hot encoding and can be revisited later with grouping or target
encoding if needed.<br><br>

**4. Secure Handling of Database Credentials:**<br>
The dataset lives on a remote MySQL server, so credentials had to be entered at
runtime (`getpass`) rather than hard-coded into the notebook.

## **16. Final Conclusion**

**1. Project Objective:**<br>
The objective was to build a Machine Learning model that pre-categorizes FicZon's
leads as High Potential or Low Potential, replacing a purely manual, post-hoc
categorization process.<br><br>

**2. Exploratory Data Analysis:**<br>
EDA on 7,422 leads surfaced clear differences in volume across lead sources, sales
agents, locations, and delivery modes, and showed the status breakdown the target
variable is built from.<br><br>

**3. Modelling:**<br>
Five classification models were trained and compared, with Random Forest tuned
further via `GridSearchCV`. Model comparison, a confusion matrix, and feature
importance analysis identified the best-performing model and the attributes driving
its predictions.<br><br>

**4. Business Impact:**<br>
The resulting model gives FicZon a way to score lead quality automatically at the
point of capture, supporting the goal of increasing sales effectiveness.<br><br>

**5. Next Steps:**<br>
Validate the `Status` → `Lead_Category` mapping with FicZon's sales team, monitor
model performance on new leads over time, and consider retraining periodically as
lead patterns evolve.

## **17. Save the Final Model for Deployment**

The best-performing model, along with the scaler, label encoder, and the exact list
of feature columns used in training, is saved to disk using `joblib`. These are the
artifacts the Flask web application will load in order to score new leads — the
column list is saved because the one-hot-encoded training features (`X.columns`)
must be reproduced in the exact same order for every new prediction.

In [ ]:
import joblib
import os

os.makedirs('model_artifacts', exist_ok=True)

joblib.dump(best_model, 'model_artifacts/lead_model.pkl')
joblib.dump(scaler, 'model_artifacts/scaler.pkl')
joblib.dump(le, 'model_artifacts/label_encoder.pkl')
joblib.dump(list(X.columns), 'model_artifacts/feature_columns.pkl')
joblib.dump(best_model_name in scaled_models, 'model_artifacts/needs_scaling.pkl')

# Save the raw category options the form will offer, taken from the training data
form_options = {
    'Source': sorted(df['Source'].unique().tolist()),
    'Sales_Agent': sorted(df['Sales_Agent'].unique().tolist()),
    'Location': sorted(df['Location'].unique().tolist()),
    'Delivery_Mode': sorted(df['Delivery_Mode'].unique().tolist()),
}
joblib.dump(form_options, 'model_artifacts/form_options.pkl')

print("Saved model artifacts:")
print(os.listdir('model_artifacts'))

## **18. Deploy as a Flask Web Application**

This section builds a small Flask web app, entirely inside this notebook, so a user
can enter a new lead's details in a browser form and instantly see the predicted
Lead Category (High Potential / Low Potential).

Because Flask normally *blocks* the cell it runs in, the app is started on a
background thread. This lets the app run live while the notebook kernel stays free.
Run the cells in this section in order, then open the printed link in your browser.

**Step 18.1 — Create the HTML template.**

Flask looks for HTML templates in a `templates/` folder next to the notebook, so it
is created first, followed by a simple form page (`index.html`) styled with basic
CSS — no external framework required.

In [ ]:
import os

os.makedirs('templates', exist_ok=True)

index_html = """
<!DOCTYPE html>
<html>
<head>
    <title>FicZon Lead Category Predictor</title>
    <style>
        body { font-family: Arial, sans-serif; background: #f4f6f8; margin: 0; padding: 40px; }
        .card { background: #fff; max-width: 520px; margin: auto; padding: 30px;
                border-radius: 10px; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }
        h2 { color: #264653; text-align: center; }
        label { display: block; margin-top: 14px; font-weight: bold; color: #333; }
        select, input { width: 100%; padding: 8px; margin-top: 6px; border-radius: 5px;
                        border: 1px solid #ccc; box-sizing: border-box; }
        button { margin-top: 22px; width: 100%; padding: 12px; background: #2a9d8f;
                 color: white; border: none; border-radius: 6px; font-size: 16px; cursor: pointer; }
        button:hover { background: #21867a; }
        .result { margin-top: 20px; padding: 14px; border-radius: 6px; text-align: center;
                  font-size: 18px; font-weight: bold; }
        .high { background: #d8f3dc; color: #1b4332; }
        .low { background: #ffe5d9; color: #9d0208; }
    </style>
</head>
<body>
    <div class="card">
        <h2>FicZon Lead Category Predictor</h2>
        <form method="POST" action="/predict">
            <label>Source</label>
            <select name="Source">
                {% for opt in options['Source'] %}<option value="{{ opt }}">{{ opt }}</option>{% endfor %}
            </select>

            <label>Sales Agent</label>
            <select name="Sales_Agent">
                {% for opt in options['Sales_Agent'] %}<option value="{{ opt }}">{{ opt }}</option>{% endfor %}
            </select>

            <label>Location</label>
            <select name="Location">
                {% for opt in options['Location'] %}<option value="{{ opt }}">{{ opt }}</option>{% endfor %}
            </select>

            <label>Delivery Mode</label>
            <select name="Delivery_Mode">
                {% for opt in options['Delivery_Mode'] %}<option value="{{ opt }}">{{ opt }}</option>{% endfor %}
            </select>

            <label>Product ID</label>
            <input type="number" step="any" name="Product_ID" value="1" required>

            <label>Mobile Number Provided?</label>
            <select name="Has_Mobile">
                <option value="1">Yes</option>
                <option value="0">No</option>
            </select>

            <label>Email Provided?</label>
            <select name="Has_Email">
                <option value="1">Yes</option>
                <option value="0">No</option>
            </select>

            <label>Lead Created Date</label>
            <input type="date" name="Created" required>

            <button type="submit">Predict Lead Category</button>
        </form>

        {% if prediction %}
        <div class="result {{ 'high' if prediction == 'High Potential' else 'low' }}">
            Predicted Category: {{ prediction }}
        </div>
        {% endif %}
    </div>
</body>
</html>
"""

with open('templates/index.html', 'w') as f:
    f.write(index_html)

print("templates/index.html created.")

**Step 18.2 — Define the Flask application.**

The app has two routes:
- `/` — renders the form, pre-filled with the real category options seen in training.
- `/predict` — reads the submitted form values, rebuilds them into the exact same
  feature format the model was trained on (same one-hot columns, same scaling), and
  returns the predicted Lead Category.

In [ ]:
from flask import Flask, request, render_template
import pandas as pd
import numpy as np
import joblib

app = Flask(__name__)

lead_model = joblib.load('model_artifacts/lead_model.pkl')
scaler_obj = joblib.load('model_artifacts/scaler.pkl')
label_enc = joblib.load('model_artifacts/label_encoder.pkl')
feature_cols = joblib.load('model_artifacts/feature_columns.pkl')
needs_scaling = joblib.load('model_artifacts/needs_scaling.pkl')
options = joblib.load('model_artifacts/form_options.pkl')


def build_feature_row(form):
    created = pd.to_datetime(form['Created'])

    raw = {
        'Product_ID': float(form['Product_ID']),
        'Created_Year': created.year,
        'Created_Month': created.month,
        'Created_Day': created.day,
        'Has_Mobile': int(form['Has_Mobile']),
        'Has_Email': int(form['Has_Email']),
        'Source': form['Source'],
        'Sales_Agent': form['Sales_Agent'],
        'Location': form['Location'],
        'Delivery_Mode': form['Delivery_Mode'],
    }

    row_df = pd.DataFrame([raw])
    row_encoded = pd.get_dummies(
        row_df, columns=['Source', 'Sales_Agent', 'Location', 'Delivery_Mode']
    )

    # Align to the exact training-time columns; any column not present for this
    # single row is filled with 0 (that category simply wasn't selected)
    row_encoded = row_encoded.reindex(columns=feature_cols, fill_value=0)

    if needs_scaling:
        row_encoded = scaler_obj.transform(row_encoded)

    return row_encoded


@app.route('/')
def home():
    return render_template('index.html', options=options, prediction=None)


@app.route('/predict', methods=['POST'])
def predict():
    features = build_feature_row(request.form)
    pred_encoded = lead_model.predict(features)[0]
    pred_label = label_enc.inverse_transform([pred_encoded])[0]
    return render_template('index.html', options=options, prediction=pred_label)


print("Flask app defined.")

**Step 18.3 — Run the app from inside the notebook.**

`app.run()` normally blocks the cell forever, which would freeze the notebook. Running
it on a background thread keeps the kernel usable while the server stays live. After
running this cell, open **http://127.0.0.1:5000** in your browser.

In [ ]:
import threading

def run_app():
    app.run(host='127.0.0.1', port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_app, daemon=True)
flask_thread.start()

print("Flask app running at: http://127.0.0.1:5000")
print("Fill in the form there to get a live Lead Category prediction.")

**Note:** The server keeps running in the background as long as this notebook's
kernel is alive. To stop it, restart the kernel (Kernel → Restart), since Flask's
development server does not expose a clean in-notebook shutdown call.